## Imports necesarios

In [ ]:
import os
import pandas as pd
import itertools
from robuspredictor import RobusPredictor   # carpeta robuspredictor/ junto a este notebook

## Carga de datos y definición de variables

In [ ]:
TRAINING_PATH = "DATOS_ENTRENAMIENTO.csv"   # Datos junto a este notebook
VALIDATION_PATH = "DATOS_VALIDACION.csv"    # Datos junto a este notebook

features = ['var1', 'var2', 'var3', 'var4', 'var5', 'var6', 'var7', 'var8', 'var9', 'var10', 'var11', 'var12', 'var13',
]
target   = "INTENSIDAD_4H"

ENTRENAMIENTO = pd.read_csv(TRAINING_PATH)
VALIDACION    = pd.read_csv(VALIDATION_PATH)

y_valid_arriendo = VALIDACION["ARRIENDO"]

X_train, y_train = ENTRENAMIENTO[features], ENTRENAMIENTO[target]
X_valid          = VALIDACION[features]

print(f"Entrenamiento: {X_train.shape[0]:,} filas x {X_train.shape[1]} variables")
print(f"Validacion   : {X_valid.shape[0]:,} filas")

## Grid Search de parámetros

Búsqueda sistemática sobre rangos de parámetros. Por cada combinación se entrena
una instancia nueva de `RobusPredictor` y se registra la Precisión Top N%.

**Flujo:**
1.  Configura los rangos y ejecuta para ver la tabla de valores,
   el total de combinaciones y el tiempo estimado. No entrena nada aún.
2.  Ejecuta el grid search completo y exporta los resultados a Excel.


### 1. Configuración y vista previa

Cada parámetro del modelo puede estar en uno de dos estados:

| Estado | GRID_PARAMS | PARAMS_FIJOS | Comportamiento |
|--------|-------------|--------------|----------------|
| **Dinámico** | activo | comentado (`##`) | el sweep explora todos sus valores |
| **Fijo** | comentado (`#`) | activo | el sweep usa siempre ese único valor |

Para pasar un parámetro de dinámico a fijo: comenta su línea en `GRID_PARAMS` y descomenta la correspondiente en `PARAMS_FIJOS`. Para el caso inverso, haz lo opuesto.

**Formatos de valor en GRID_PARAMS:**

| Formato | Ejemplo | Valores generados |
|---------|---------|-------------------|
| `(inicio, fin, paso)` | `(100, 300, 50)` | 100, 150, 200, 250, 300 |
| `[v1, v2, ...]` | `[16.0, 30.0]` | 16.0 y 30.0 |
| `[v]` | `[2]` | solo 2 (equivale a fijo) |

La tupla debe tener **siempre exactamente 3 elementos**. Para valores explícitos usa lista. Fijarse en uso de parentesis o corchetes para uno u otro.

> **⚙️ Modifica `GRID_PARAMS` y `PARAMS_FIJOS`**, luego ejecuta esta celda para ver cuántas combinaciones se generarán antes de correr el search.

In [ ]:
# Configuración del grid search

GRID_PARAMS = { # Dejar o sacar comentados los parámetros que quieras fijar y no explorar en el grid search

    "n_min":   [5, 10, 15],
    "n_max":   [20, 30, 50],
    "std_min": [0.0, 5.0],
    "std_max": [10.0, 20.0],
    "mean_min":[0.00, 0.05, 0.10],
    "mean_max":[24.0, 25.0, 26.0],
    "n_dom":   [2,3],
}

PARAMS_FIJOS = dict(   # Dejar o Sacar los comentarios de los parámetros que quieras fijar y no explorar en el grid search
    #n_min             = 20,     
    #n_max             = 500,    
    n_dom             = 2,      
    #std_min           = 0.0,    
    #std_max           = 8.0,    
    #mean_min          = 0.05,   
    #mean_max          = 24.0,   
    default_value     = 0,
    verbose           = False,
    use_default_value = True,
)

TOP_PCT           = 0.05   # 0.05 = Top 5% | 0.01 = Top 1% | 0.10 = Top 10%
SEG_POR_ITERACION = 30     # segundos estimados por iteración (ajusta según tu equipo)

# ── Expansión y validación ────────────────────────────────────────────────────
def _expandir(val, nombre=""):
    if isinstance(val, tuple):
        if len(val) != 3:
            raise ValueError(
                f"Parámetro '{nombre}': tupla incorrecta {val}.\n"
                f"  Usa exactamente 3 elementos: (inicio, fin, paso).\n"
                f"  Para valores explícitos usa lista: [{val[0]}, {val[-1]}]"
            )
        inicio, fin, paso = val
        usar_int = all(isinstance(x, int) and not isinstance(x, bool)
                       for x in (inicio, fin, paso))
        vals, v = [], inicio
        while v <= fin + 1e-9:
            vals.append(int(round(v)) if usar_int else round(float(v), 8))
            v += paso
        return vals
    return val if isinstance(val, list) else [val]

grid_expandido = {k: _expandir(v, k) for k, v in GRID_PARAMS.items()}

# ── Vista previa ──────────────────────────────────────────────────────────────
filas_tabla, producto_total = [], 1
for param, vals in grid_expandido.items():
    tipo = "rango (inicio,fin,paso)" if isinstance(GRID_PARAMS[param], tuple) else "lista explícita"
    filas_tabla.append({
        "parámetro": param, "formato": tipo,
        "definición": str(GRID_PARAMS[param]),
        "valores generados": str(vals), "N": len(vals),
    })
    producto_total *= len(vals)

display(pd.DataFrame(filas_tabla).set_index("parámetro"))
print()

# ── Combinaciones válidas ─────────────────────────────────────────────────────
keys   = list(grid_expandido.keys())
combos = []
for _vals in itertools.product(*grid_expandido.values()):
    _combo = dict(zip(keys, _vals))
    _full  = {**PARAMS_FIJOS, **_combo}
    if _full.get("n_max", float("inf")) >= _full.get("n_min", 0):
        combos.append(_combo)

n_bruto  = producto_total
n_valido = len(combos)
n_filtro = n_bruto - n_valido

t_seg  = n_valido * SEG_POR_ITERACION
t_min  = t_seg // 60
t_hora = t_min // 60
t_str  = (f"{t_hora}h {t_min % 60}m" if t_hora > 0 else f"{t_min}m {t_seg % 60}s")

print("RESUMEN DE COMBINACIONES")
print("=" * 70)
multiplicacion = " × ".join(f"{len(v)} ({k})" for k, v in grid_expandido.items())
print(f"  Producto total          : {multiplicacion}")
print(f"  Combinaciones brutas    : {n_bruto:,}")
print(f"  Filtradas (n_max<n_min) : {n_filtro:,}")
print(f"  Combinaciones válidas   : {n_valido:,}  ← estas se ejecutarán")
print(f"  Tiempo proyectado       : {t_str}  (@ {SEG_POR_ITERACION}s por iteración)")
print()
print("  → Ejecuta la celda siguiente cuando estés conforme con esta configuración.")

### 2. Ejecutar grid search y exportar resultados

Por cada combinación válida: entrena un `RobusPredictor`, genera predicciones sobre
validación, calcula Precisión Top N% y registra cobertura y cubos estables.

Al finalizar exporta tres hojas Excel:
- **resultados** — todas las combinaciones ordenadas por precisión
- **errores** — combinaciones que fallaron (si las hay)
- **configuracion** — parámetros sweep y fijos usados en esta corrida

>Ejecutar **solo después** de haber revisado la celda anterior

In [ ]:
import warnings
import time
from datetime import datetime
warnings.filterwarnings("ignore")

prec_col        = f"precision_top{int(TOP_PCT*100)}pct"
resultados_grid = []
_NO_TRACK       = {"verbose", "random_state", "use_default_value", "default_value"}

print(f"Grid search iniciado: {n_valido} combinaciones  |  Top {int(TOP_PCT*100)}%")
print(f"Tiempo proyectado   : {t_str}  (@ {SEG_POR_ITERACION}s/iter estimado)")
print("-" * 75)
t_gs_inicio = time.perf_counter()

for i, combo in enumerate(combos, start=1):
    params_iter = {**PARAMS_FIJOS, **combo}
    precision_i = cobertura_i = n_est_i = error_i = None

    try:
        m = RobusPredictor(**params_iter)
        m.fit(X_train, y_train)
        y_p         = m.predict(X_valid)
        precision_i = m.best_percentage(y_valid_arriendo, top_pct=TOP_PCT)
        n_est_i     = len(m.stable_cubes)
        n_cub       = (y_p != params_iter.get("default_value", 0)).sum() if params_iter.get("use_default_value", True) else len(y_p)
        cobertura_i = n_cub / len(y_p) * 100
        status      = f"prec={precision_i:.4f}  cob={cobertura_i:.1f}%  est={n_est_i}"
    except Exception as e:
        error_i = str(e)
        status  = f"ERROR: {error_i[:60]}"

    params_row = {k: v for k, v in params_iter.items() if k not in _NO_TRACK}
    resultados_grid.append({
        "iter": i, **params_row,
        prec_col:    round(precision_i, 4) if precision_i is not None else None,
        "cob_%":     round(cobertura_i,  1) if cobertura_i  is not None else None,
        "cubos_est": n_est_i,
        "error":     error_i,
    })

    pct_done  = i / n_valido * 100
    param_str = "  ".join(f"{k}={v}" for k, v in params_row.items())
    print(f"  [{i:3d}/{n_valido}  {pct_done:4.0f}%]  {param_str}  → {status}")

t_gs_total = time.perf_counter() - t_gs_inicio

t_real_m   = int(t_gs_total // 60)
t_real_s   = t_gs_total % 60
t_real_str = f"{t_real_m}m {t_real_s:.0f}s" if t_real_m > 0 else f"{t_real_s:.1f}s"

print("-" * 75)
print(f"Tiempo proyectado   : {t_str}  (@ {SEG_POR_ITERACION}s/iter estimado)")
print(f"Tiempo real         : {t_real_str}  ({t_gs_total/n_valido:.1f}s/iter promedio)")

df_grid = pd.DataFrame(resultados_grid)
n_err   = df_grid["error"].notna().sum()
df_ok   = df_grid[df_grid["error"].isna()].copy()

if df_ok[prec_col].notna().any():
    df_ok = df_ok.sort_values(prec_col, ascending=False).reset_index(drop=True)

param_cols = [k for k in (list(resultados_grid[0].keys()) if resultados_grid else [])
              if k not in _NO_TRACK and k not in {"iter", prec_col, "cob_%", "cubos_est", "error"}]
cols_vis = ["iter"] + param_cols + [prec_col, "cob_%", "cubos_est"]
cols_vis = [c for c in cols_vis if c in df_ok.columns]

print(f"\nTop 10 combinaciones por {prec_col}:")
display(df_ok[cols_vis].head(10))

if n_err:
    print(f"\nX {n_err} combinación(es) con error — ver columna 'error' en el Excel.")

ts        = datetime.now().strftime("%d%m%Y%H%M")
GRID_PATH = f"resultados_grid_search_{ts}.xlsx"

with pd.ExcelWriter(GRID_PATH, engine="openpyxl") as writer:
    df_ok.sort_values(prec_col, ascending=False).to_excel(
        writer, sheet_name="resultados", index=False)
    if n_err:
        df_grid[df_grid["error"].notna()].to_excel(
            writer, sheet_name="errores", index=False)
    config_rows = (
        [{"parámetro": k, "tipo": "sweep",  "valores": str(v)} for k, v in grid_expandido.items()] +
        [{"parámetro": k, "tipo": "fijo",   "valores": str(v)} for k, v in PARAMS_FIJOS.items()]
    )
    pd.DataFrame(config_rows).to_excel(writer, sheet_name="configuracion", index=False)

print(f"\nResultados exportados en: {GRID_PATH}")
if df_ok[prec_col].notna().any():
    mejor = df_ok.iloc[0]
    print(f"\nMejor combinación (Top {int(TOP_PCT*100)}% = {mejor[prec_col]:.4f}):")
    for col in param_cols + ["cob_%", "cubos_est"]:
        if col in mejor.index:
            print(f"  {col:<15} = {mejor[col]}")